<a href="https://colab.research.google.com/github/R341CUL/python-together/blob/main/08-machine-learning/09_kaggle_workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 9 · The Kaggle workflow — how to attack a real dataset

[Kaggle](https://www.kaggle.com) is where people compete to build the best model
on a shared dataset. Even if you never compete, its *workflow* is the
professional template for any ML problem. We'll run the whole loop on a built-in
dataset (so it works offline), but the steps are identical for a real Kaggle CSV.

> On real Kaggle you'd start with `df = pd.read_csv("train.csv")`. Here we load a
> bundled dataset so the lesson runs anywhere.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer

## Step 1 — Load and *look* (EDA)

Exploratory Data Analysis = understand the data before modelling. Shape, the
target balance, missing values, a peek at the rows.

In [ ]:
data = load_breast_cancer(as_frame=True)
df = data.frame
print("shape:", df.shape)
print("\ntarget balance:\n", df["target"].value_counts())
print("\nany missing values?", df.isna().any().any())
df.head(3)

## Step 2 — Split features `X` and target `y`, then hold out a test set

The cardinal rule: the test set is sacred. You don't look at it until the end.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["target"])
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=0
)
print("train:", X_train.shape, " test:", X_test.shape)

## Step 3 — A dumb baseline first

Always establish the score to beat. If you can't beat "always guess the most
common class", something is wrong.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("baseline accuracy:", round(accuracy_score(y_test, baseline.predict(X_test)), 3))

## Step 4 — A real model, scored with cross-validation

We use a pipeline (lesson 5) and cross-validation so the score is trustworthy
*before* we ever touch the test set.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

model = Pipeline([
    ("scale", StandardScaler()),
    ("rf", RandomForestClassifier(n_estimators=200, random_state=0)),
])
cv = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy")
print("cross-val accuracy:", cv.round(3), " mean:", round(cv.mean(), 3))

## Step 5 — Which features mattered?

Interpreting the model builds trust (and ideas for better features). Random
forests expose feature importances for free.

In [ ]:
model.fit(X_train, y_train)
importances = pd.Series(
    model.named_steps["rf"].feature_importances_, index=X.columns
).sort_values(ascending=False)
print("top 5 features:\n", importances.head(5).round(3))

## Step 6 — FINAL evaluation on the held-out test set (once!)

In [ ]:
test_acc = accuracy_score(y_test, model.predict(X_test))
print("final test accuracy:", round(test_acc, 3), " vs baseline:",
      round(accuracy_score(y_test, baseline.predict(X_test)), 3))

On a real competition, Step 7 is to predict on Kaggle's unlabeled `test.csv` and
write a `submission.csv`:

```python
preds = model.predict(kaggle_test)
pd.DataFrame({"id": ids, "target": preds}).to_csv("submission.csv", index=False)
```

## 🏋️ Your turn

Swap the `RandomForestClassifier` for `LogisticRegression(max_iter=5000)` inside
the pipeline and compare cross-val accuracy. Then look at the top features — do
the two models agree on what matters?

In [ ]:
# from sklearn.linear_model import LogisticRegression
# ... build a new pipeline and compare cross_val_score ...

### ✅ What you learned — the universal ML loop
1. **EDA**: understand the data (shape, balance, missing).
2. **Split** off a sacred test set.
3. **Baseline** to set the bar.
4. **Model + cross-validation** for a trustworthy score.
5. **Interpret** (feature importance).
6. **Final test** once, then **submit**.

🎉 That's the machine-learning track. Now go build in `10-projects/`.